# Part 1: Data Ingestion
## Programmatic Download (5 marks)

In [1]:
pip install pandas scikit-learn matplotlib seaborn joblib torch shap

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install shap

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Scikit-learn: Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Scikit-learn: Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import *
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

import numpy as np

import shap

import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score, classification_report
)

In [4]:
import os
import sys
import requests
import pandas as pd
import polars as pl
import duckdb
from datetime import datetime
import time
from pathlib import Path
import requests

In [5]:
data_dir = Path("data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

trip_data_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
trip_data_file = data_dir / "yellow_tripdata_2024-01.parquet"

zone_data_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
zone_data_file = data_dir / "taxi_zone_lookup.csv"

In [6]:
def download_file(url, file_path):
    if not file_path.exists():
        print(f"Downloading from: {url}")
        response = requests.get(url)
        
        with open(file_path, 'wb') as f:
            f.write(response.content)
        
        file_size_mb = os.path.getsize(file_path) / 1e6
        print(f"Downloaded: {file_path.name} ({file_size_mb:.1f} MB)")
        return True
    else:
        file_size_mb = os.path.getsize(file_path) / 1e6
        print(f"File already exists: {file_path.name} ({file_size_mb:.1f} MB)")
        return False

In [7]:
download_file(trip_data_url, trip_data_file)

download_file(zone_data_url, zone_data_file)


File already exists: yellow_tripdata_2024-01.parquet (50.0 MB)
File already exists: taxi_zone_lookup.csv (0.0 MB)


False

# Part 2: Data Transformation & Analysis (30 marks)

## 4. Data Cleaning (10 marks)

### e) Removing rows with null values in critical columns (pickup/dropoff times, locations, fare) 

In [8]:
df = pl.read_parquet(trip_data_file)
cur = df.height

print(f"This is the data before dropping null vaules: {cur}")

clean = df.drop_nulls(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "fare_amount"])
print(f"This is the data after dropping null vaules: {clean.height}")
print(f"Amount dropped: {cur - clean.height}")

This is the data before dropping null vaules: 2964624
This is the data after dropping null vaules: 2964624
Amount dropped: 0


### f) Filtering out invalid trips: trips with zero or negative distance, negative fares, or fares exceeding $500 

In [9]:
# Filtering out invalid trips: trips with zero or negative distance
cur = clean.height
print(f"The current rows before removing invaild distances are {cur}")
clean = clean.filter(pl.col("trip_distance").is_not_null() & (pl.col("trip_distance") > 0))
invalid_distances = cur - clean.height
print(f"The cleaned distances are {invalid_distances}")

# Filtering out invalid trips: trips with negative fares, or fares exceeding $500 
cur = clean.height
print(f"The current rows before removing invaild fares are {cur}")
clean = clean.filter((pl.col('fare_amount') > 0) & (pl.col('fare_amount') < 500))
invalid_fares = cur - clean.height
print(f"The cleaned fares are {invalid_fares}")

# Removing trips where dropoff time is before pickup time 
cur = clean.height
print(f"The current rows before removing invaild dates are {cur}")
clean = clean.filter(pl.col('tpep_dropoff_datetime') > pl.col('tpep_pickup_datetime'))
invalid_dates = cur - clean.height
print(f"The cleaned dates are {invalid_dates}")

cur = clean.height
clean = clean.filter(pl.col("payment_type") == 1)
credit_card_only = cur - clean.height
print(f"Removed {credit_card_only:,} non-credit-card rows, {clean.height:,} rows remaining")


The current rows before removing invaild distances are 2964624
The cleaned distances are 60371
The current rows before removing invaild fares are 2904253
The cleaned fares are 34573
The current rows before removing invaild dates are 2869680
The cleaned dates are 112
Removed 571,221 non-credit-card rows, 2,298,347 rows remaining


## Feature Engineering (10 marks)

In [10]:
clean = clean.with_columns([
    # i) trip_duration_minutes: calculated from pickup and dropoff timestamps
    (
        (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
        .dt.total_seconds() / 60).alias("trip_duration_minutes"),

    # j) trip_speed_mph: distance divided by duration (handle division by zero) 
    pl.when(
        ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
         .dt.total_seconds()) > 0
    )
    .then(
        pl.col("trip_distance") /
        (((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
          .dt.total_seconds()) / 3600)
    )
    .otherwise(None)
    .alias("trip_speed_mph"),

    # k) pickup_hour: hour of day (0-23) extracted from pickup timestamp
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),

    # l) pickup_day_of_week: day name (Monday-Sunday) extracted from pickup timestamp
    pl.col("tpep_pickup_datetime").dt.strftime("%A").alias("pickup_day_of_week")
])


# Assignment 2

In [11]:
clean

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_minutes,trip_speed_mph,pickup_hour,pickup_day_of_week
i32,datetime[ns],datetime[ns],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8,str
1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.8,1,"""N""",140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.0,6.6,16.363636,0,"""Monday"""
1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.7,1,"""N""",236,79,1,23.3,3.5,0.5,3.0,0.0,1.0,31.3,2.5,0.0,17.916667,15.739535,0,"""Monday"""
1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.4,1,"""N""",79,211,1,10.0,3.5,0.5,2.0,0.0,1.0,17.0,2.5,0.0,8.3,10.120482,0,"""Monday"""
1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.8,1,"""N""",211,148,1,7.9,3.5,0.5,3.2,0.0,1.0,16.1,2.5,0.0,6.1,7.868852,0,"""Monday"""
1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.7,1,"""N""",148,141,1,29.6,3.5,0.5,6.9,0.0,1.0,41.5,2.5,0.0,32.383333,8.708183,0,"""Monday"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,2024-01-31 23:15:08,2024-01-31 23:29:33,1,7.68,1,"""N""",230,243,1,31.0,1.0,0.5,7.2,0.0,1.0,43.2,2.5,0.0,14.416667,31.963006,23,"""Wednesday"""
2,2024-01-31 23:10:28,2024-01-31 23:18:30,1,3.51,1,"""N""",138,129,1,16.3,6.0,0.5,4.76,0.0,1.0,30.31,0.0,1.75,8.033333,26.215768,23,"""Wednesday"""
2,2024-01-31 23:01:04,2024-01-31 23:17:35,1,3.36,1,"""N""",162,261,1,18.4,1.0,0.5,5.85,0.0,1.0,29.25,2.5,0.0,16.516667,12.205853,23,"""Wednesday"""


In [12]:
clean = clean.to_pandas()

# Part 1: Data Preprocessing & Feature Engineering (25 marks)

##  Feature Engineering (10 marks)

### a) Temporal features

In [13]:
clean["tpep_pickup_datetime"] = pd.to_datetime(clean["tpep_pickup_datetime"])
clean["pickup_day_of_week"] = clean["tpep_pickup_datetime"].dt.dayofweek 
clean["is_weekend"] = clean["pickup_day_of_week"] >= 5

### b) Trip features

In [14]:
clean["log_trip_distance"] = np.log1p(clean["trip_distance"])

### c) Fare features

In [15]:
clean["fare_per_mile"] = clean["fare_amount"] / clean["trip_distance"].replace(0,1)
clean["fare_per_minute"] = clean["fare_amount"] / clean["trip_duration_minutes"].replace(0,1)

### d) Zone Features

In [16]:
zones = pd.read_csv(zone_data_file, usecols=["LocationID", "Borough"])

# Join pickup borough
clean = clean.merge(
    zones.rename(columns={"LocationID": "PULocationID", "Borough": "pickup_borough"}),
    on="PULocationID",
    how="left"
)

# Join dropoff borough
clean = clean.merge(
    zones.rename(columns={"LocationID": "DOLocationID", "Borough": "dropoff_borough"}),
    on="DOLocationID",
    how="left"
)

In [17]:
clean = pd.get_dummies(clean, columns=["pickup_borough", "dropoff_borough"])
clean = clean.astype({col: "int" for col in clean.columns if "borough" in col})

## Target Variable Creation (5 marks)

In [18]:
# a) tip_amount (continuous, for regression) 
reg_target = clean["tip_amount"]

# b) high_tip (binary: 1 if tip_amount > 20% of fare_amount, 0 otherwise, for classification) 
clean["high_tip"] = (clean["tip_amount"] > 0.20 * clean["fare_amount"]).astype(int)


## 3. Data Splitting & Scaling (10 marks)

### a) Split data into training (70%), validation (15%), and test (15%) sets using stratified sampling for the classification target 

In [19]:
excluded_features = [
    "tip_amount",
    "high_tip",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "total_amount",
    "store_and_fwd_flag",
    "payment_type",      
]

feature_cols = [col for col in clean.columns if col not in excluded_features]

X = clean[feature_cols]
y_reg = clean["tip_amount"]
y_clf = clean["high_tip"]

# First split - 70% train, 30% temp
X_train, X_temp, y_train_clf, y_temp_clf, y_train_reg, y_temp_reg = train_test_split(
    X, y_clf, y_reg, test_size=0.30, stratify=y_clf, random_state=42
)

# Second split - 15% validation, 15% test
X_val, X_test, y_val_clf, y_test_clf, y_val_reg, y_test_reg = train_test_split(
    X_temp, y_temp_clf, y_temp_reg, test_size=0.50, stratify=y_temp_clf, random_state=42
)

print(f"Training samples:   {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")
print(f"Test samples:       {len(X_test):,}")
print(f"\nClass distribution (high_tip):")
print(f"  Train - 0: {(y_train_clf == 0).sum():,}  1: {(y_train_clf == 1).sum():,}")
print(f"  Val   - 0: {(y_val_clf   == 0).sum():,}  1: {(y_val_clf   == 1).sum():,}")
print(f"  Test  - 0: {(y_test_clf  == 0).sum():,}  1: {(y_test_clf  == 1).sum():,}")

Training samples:   1,608,842
Validation samples: 344,752
Test samples:       344,753

Class distribution (high_tip):
  Train - 0: 387,205  1: 1,221,637
  Val   - 0: 82,973  1: 261,779
  Test  - 0: 82,973  1: 261,780


### b) Apply appropriate scaling (StandardScaler or MinMaxScaler) to numeric features; fit on training data only 

In [20]:
X_train

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,fare_amount,extra,mta_tax,tolls_amount,...,pickup_borough_Queens,pickup_borough_Staten Island,pickup_borough_Unknown,dropoff_borough_Bronx,dropoff_borough_Brooklyn,dropoff_borough_EWR,dropoff_borough_Manhattan,dropoff_borough_Queens,dropoff_borough_Staten Island,dropoff_borough_Unknown
2106172,1,1,0.80,1,263,236,10.0,5.0,0.5,0.00,...,0,0,0,0,0,0,1,0,0,0
2206115,2,1,2.16,1,161,236,10.7,1.0,0.5,0.00,...,0,0,0,0,0,0,1,0,0,0
388649,2,1,0.50,1,236,263,5.8,1.0,0.5,0.00,...,0,0,0,0,0,0,1,0,0,0
1969948,2,2,18.64,2,132,163,70.0,0.0,0.5,6.94,...,1,0,0,0,0,0,1,0,0,0
1292751,1,1,1.90,1,262,163,14.9,2.5,0.5,0.00,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1745938,2,1,3.57,1,68,13,21.2,0.0,0.5,0.00,...,0,0,0,0,0,0,1,0,0,0
1956759,2,1,1.71,1,90,170,18.4,0.0,0.5,0.00,...,0,0,0,0,0,0,1,0,0,0
1110500,2,1,0.41,1,163,48,4.4,1.0,0.5,0.00,...,0,0,0,0,0,0,1,0,0,0
1093680,2,1,1.41,1,238,41,7.9,2.5,0.5,0.00,...,0,0,0,0,0,0,1,0,0,0


In [21]:
numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# fit_transform on train only, then transform val and test
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

### c) Document the number of samples in each split and the class distribution of high_tip in each split 

In [22]:
total = len(X_train) + len(X_val) + len(X_test)

print("SPLIT SIZES")
print("=" * 45)
print(f"Training samples:   {len(X_train):,} ({len(X_train)/total*100:.1f}%)")
print(f"Validation samples: {len(X_val):,} ({len(X_val)/total*100:.1f}%)")
print(f"Test samples:       {len(X_test):,} ({len(X_test)/total*100:.1f}%)")
print(f"Total:              {total:,}")

print("\nCLASS DISTRIBUTION (high_tip)")
print("=" * 45)
for name, y in [("Train", y_train_clf), ("Validation", y_val_clf), ("Test", y_test_clf)]:
    total_split = len(y)
    count_0 = (y == 0).sum()
    count_1 = (y == 1).sum()
    print(f"\n{name}:")
    print(f"  Low tip  (0): {count_0:,} ({count_0/total_split*100:.1f}%)")
    print(f"  High tip (1): {count_1:,} ({count_1/total_split*100:.1f}%)")

SPLIT SIZES
Training samples:   1,608,842 (70.0%)
Validation samples: 344,752 (15.0%)
Test samples:       344,753 (15.0%)
Total:              2,298,347

CLASS DISTRIBUTION (high_tip)

Train:
  Low tip  (0): 387,205 (24.1%)
  High tip (1): 1,221,637 (75.9%)

Validation:
  Low tip  (0): 82,973 (24.1%)
  High tip (1): 261,779 (75.9%)

Test:
  Low tip  (0): 82,973 (24.1%)
  High tip (1): 261,780 (75.9%)


### d) Print a summary of feature names, types, and any features excluded from modeling (and why) 

In [23]:
excluded_reasons = {
    "tip_amount":             "Regression target variable",
    "high_tip":               "Classification target variable",
    "tpep_pickup_datetime":   "Raw datetime",
    "tpep_dropoff_datetime":  "Raw datetime",
    "total_amount":           "Leakage — directly includes tip_amount in its calculation",
}

print("EXCLUDED FEATURES")
print("=" * 65)
for col, reason in excluded_reasons.items():
    print(f"\n  {col}")
    print(f"    Reason: {reason}")

print("INCLUDED FEATURES")
print("=" * 65)
print(f"{'#':<5} {'Feature':<35} {'Dtype':<15}")
print("-" * 65)
for i, col in enumerate(X_train_scaled.columns, 1):
    dtype = str(X_train_scaled[col].dtype)
    print(f"{i:<5} {col:<35} {dtype:<15}")

print("-" * 65)
print(f"\nTotal features used for modeling: {X_train_scaled.shape[1]}")

EXCLUDED FEATURES

  tip_amount
    Reason: Regression target variable

  high_tip
    Reason: Classification target variable

  tpep_pickup_datetime
    Reason: Raw datetime

  tpep_dropoff_datetime
    Reason: Raw datetime

  total_amount
    Reason: Leakage — directly includes tip_amount in its calculation
INCLUDED FEATURES
#     Feature                             Dtype          
-----------------------------------------------------------------
1     VendorID                            float64        
2     passenger_count                     float64        
3     trip_distance                       float64        
4     RatecodeID                          float64        
5     PULocationID                        float64        
6     DOLocationID                        float64        
7     fare_amount                         float64        
8     extra                               float64        
9     mta_tax                             float64        
10    tolls_amount       

# Part 2: Model Training & Tuning (30 marks) 

## Baseline Models (10 marks)

### a) Regression: Train a Linear Regression and a Random Forest Regressor to predict tip_amount 

In [24]:
imputer = SimpleImputer(strategy="median")

X_train_scaled = pd.DataFrame(
    imputer.fit_transform(X_train_scaled),
    columns=X_train_scaled.columns
)
X_val_scaled = pd.DataFrame(
    imputer.transform(X_val_scaled),
    columns=X_val_scaled.columns
)
X_test_scaled = pd.DataFrame(
    imputer.transform(X_test_scaled),
    columns=X_test_scaled.columns
)

In [25]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train_reg)

# Random Forest Regression
rf_reg = RandomForestRegressor(n_estimators=50, max_depth=20, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_scaled, y_train_reg)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",50
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples a

In [26]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train_reg)

rf_reg = RandomForestRegressor(n_estimators=50, max_depth=20, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_scaled, y_train_reg)

lr_preds  = lr.predict(X_val_scaled)
rf_preds  = rf_reg.predict(X_val_scaled)


### b) Classification: Train a Logistic Regression and a Random Forest Classifier to predict high_tip

In [27]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train_clf)

rf_clf = RandomForestClassifier(n_estimators=50, max_depth=20, random_state=42, n_jobs=-1)
rf_clf.fit(X_train_scaled, y_train_clf)

log_preds = log_reg.predict(X_val_scaled)
rf_preds  = rf_clf.predict(X_val_scaled)


### c) Report performance on the validation set for each model. For regression, report MAE, RMSE, and R². For classification, report accuracy, precision, recall, F1-score, and AUC-ROC 

In [28]:
# Regression Predictions
lr_preds    = lr.predict(X_val_scaled)
rf_reg_preds = rf_reg.predict(X_val_scaled)

print("REGRESSION PERFORMANCE (Validation Set)")
print("=" * 55)
print(f"{'Metric':<12} {'Linear Regression':>20} {'Random Forest':>20}")
print("-" * 55)
for name, preds in [("Linear Regression", lr_preds), ("Random Forest", rf_reg_preds)]:
    mae  = mean_absolute_error(y_val_reg, preds)
    rmse = np.sqrt(mean_squared_error(y_val_reg, preds))
    r2   = r2_score(y_val_reg, preds)
print(f"{'MAE':<12} {mean_absolute_error(y_val_reg, lr_preds):>20.4f} {mean_absolute_error(y_val_reg, rf_reg_preds):>20.4f}")
print(f"{'RMSE':<12} {np.sqrt(mean_squared_error(y_val_reg, lr_preds)):>20.4f} {np.sqrt(mean_squared_error(y_val_reg, rf_reg_preds)):>20.4f}")
print(f"{'R²':<12} {r2_score(y_val_reg, lr_preds):>20.4f} {r2_score(y_val_reg, rf_reg_preds):>20.4f}")



# Classification Predictions 
log_preds    = log_reg.predict(X_val_scaled)
rf_clf_preds = rf_clf.predict(X_val_scaled)
log_proba    = log_reg.predict_proba(X_val_scaled)[:, 1]
rf_proba     = rf_clf.predict_proba(X_val_scaled)[:, 1]

print("CLASSIFICATION PERFORMANCE (Validation Set)")
print("=" * 55)
print(f"{'Metric':<12} {'Logistic Regression':>20} {'Random Forest':>20}")
print("-" * 55)
print(f"{'Accuracy':<12} {accuracy_score(y_val_clf, log_preds):>20.4f} {accuracy_score(y_val_clf, rf_clf_preds):>20.4f}")
print(f"{'Precision':<12} {precision_score(y_val_clf, log_preds):>20.4f} {precision_score(y_val_clf, rf_clf_preds):>20.4f}")
print(f"{'Recall':<12} {recall_score(y_val_clf, log_preds):>20.4f} {recall_score(y_val_clf, rf_clf_preds):>20.4f}")
print(f"{'F1-Score':<12} {f1_score(y_val_clf, log_preds):>20.4f} {f1_score(y_val_clf, rf_clf_preds):>20.4f}")
print(f"{'AUC-ROC':<12} {roc_auc_score(y_val_clf, log_proba):>20.4f} {roc_auc_score(y_val_clf, rf_proba):>20.4f}")

REGRESSION PERFORMANCE (Validation Set)
Metric          Linear Regression        Random Forest
-------------------------------------------------------
MAE                        1.2072               1.1881
RMSE                       2.3854               2.3734
R²                         0.6180               0.6219
CLASSIFICATION PERFORMANCE (Validation Set)
Metric        Logistic Regression        Random Forest
-------------------------------------------------------
Accuracy                   0.7709               0.7711
Precision                  0.7691               0.7694
Recall                     0.9979               0.9976
F1-Score                   0.8687               0.8687
AUC-ROC                    0.6100               0.6248


## Hyperparameter Tuning (10 marks)

In [29]:
param_dist = {
    "n_estimators":      [50, 100, 200, 300],
    "max_depth":         [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2"]
}

rf_reg_tune = RandomForestRegressor(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=rf_reg_tune,
    param_distributions=param_dist,
    n_iter=20,
    scoring="r2",
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)


In [ ]:
X_tune, _, y_tune_reg, _ = train_test_split(
    X_train_scaled,
    y_train_reg,
    train_size=200_000,
    random_state=42
)

print(f"Tuning sample size: {len(X_tune):,}")

random_search.fit(X_tune, y_tune_reg)


Tuning sample size: 200,000
Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [ ]:
print("HYPERPARAMETER SEARCH SPACE")
print("=" * 50)
for param, values in param_dist.items():
    print(f"  {param:<25} {values}")

print("BEST HYPERPARAMETERS FOUND")
print("=" * 50)
for param, value in random_search.best_params_.items():
    print(f"  {param:<25} {value}")
print(f"\n  Best CV R²: {random_search.best_score_:.4f}")

best_rf_reg = RandomForestRegressor(
    **random_search.best_params_,
    random_state=42,
    n_jobs=-1
)
best_rf_reg.fit(X_train_scaled, y_train_reg)


In [ ]:
# Baseline RF Regressor
baseline_reg_preds = rf_reg.predict(X_val_scaled)

# Tuned RF Regressor
tuned_reg_preds = best_rf_reg.predict(X_val_scaled)

comparison_df = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Baseline RF": [
        mean_absolute_error(y_val_reg, baseline_reg_preds),
        np.sqrt(mean_squared_error(y_val_reg, baseline_reg_preds)),
        r2_score(y_val_reg, baseline_reg_preds),
    ],
    "Tuned RF": [
        mean_absolute_error(y_val_reg, tuned_reg_preds),
        np.sqrt(mean_squared_error(y_val_reg, tuned_reg_preds)),
        r2_score(y_val_reg, tuned_reg_preds),
    ]
})

comparison_df["Change"] = (comparison_df["Tuned RF"] - comparison_df["Baseline RF"]).map("{:+.4f}".format)
comparison_df = comparison_df.round(5)
comparison_df


## Neural Network Model (10 marks)

### a) Design a feedforward neural network with at least 2 hidden layers for either the regression or classification task (your choice) 

In [ ]:
print(f'PyTorch version: {torch.__version__}') 
print(f'CUDA available: {torch.cuda.is_available()}') 
if torch.cuda.is_available(): 
    print(f'GPU: {torch.cuda.get_device_name(0)}') 
    device = torch.device('cuda') 
else: 
    print('Using CPU - training will be slower') 
    device = torch.device('cpu')

In [ ]:
import torch
import torch.nn as nn

class TipClassifier(nn.Module):
    def __init__(self, input_dim):
        super(TipClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)   
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

input_dim = X_train_scaled.shape[1]
model = TipClassifier(input_dim)
print(model)
print(f"\nInput features: {input_dim}")

### b) Implement proper training with: batch processing using DataLoader, an appropriate loss function (MSELoss for regression or BCEWithLogitsLoss for classification), an optimizer (Adam or SGD), and training for at least 20 epochs 

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

# Convert to tensors
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_clf.values,    dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val_scaled.values,   dtype=torch.float32)
y_val_tensor   = torch.tensor(y_val_clf.values,      dtype=torch.float32)

# b) DataLoader with batch processing
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,   y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=1024, shuffle=False)

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 20
train_losses, val_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    batch_losses = []
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())

    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            val_batch_losses.append(loss.item())

    train_loss = np.mean(batch_losses)
    val_loss   = np.mean(val_batch_losses)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1:>2}/{EPOCHS}  Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}")

### c) Plot training and validation loss curves across epochs 

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label="Train Loss",      marker="o")
plt.plot(range(1, EPOCHS + 1), val_losses,   label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Neural Network — Training vs Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()

### d) Report the same evaluation metrics as the Scikit-learn models for comparison 

In [ ]:
model.eval()
with torch.no_grad():
    logits    = model(X_val_tensor)
    proba_nn  = torch.sigmoid(logits).numpy()
    preds_nn  = (proba_nn >= 0.5).astype(int)

nn_accuracy  = accuracy_score(y_val_clf,  preds_nn)
nn_precision = precision_score(y_val_clf, preds_nn)
nn_recall    = recall_score(y_val_clf,    preds_nn)
nn_f1        = f1_score(y_val_clf,        preds_nn)
nn_auc       = roc_auc_score(y_val_clf,   proba_nn)

# Baseline 
log_preds_val = log_reg.predict(X_val_scaled)
log_proba_val = log_reg.predict_proba(X_val_scaled)[:, 1]
rf_preds_val  = rf_clf.predict(X_val_scaled)
rf_proba_val  = rf_clf.predict_proba(X_val_scaled)[:, 1]

comparison_df = pd.DataFrame([
    {
        "Model":     "Logistic Regression",
        "Accuracy":  accuracy_score(y_val_clf,  log_preds_val),
        "Precision": precision_score(y_val_clf, log_preds_val),
        "Recall":    recall_score(y_val_clf,    log_preds_val),
        "F1-Score":  f1_score(y_val_clf,        log_preds_val),
        "AUC-ROC":   roc_auc_score(y_val_clf,   log_proba_val)
    },
    {
        "Model":     "Random Forest",
        "Accuracy":  accuracy_score(y_val_clf,  rf_preds_val),
        "Precision": precision_score(y_val_clf, rf_preds_val),
        "Recall":    recall_score(y_val_clf,    rf_preds_val),
        "F1-Score":  f1_score(y_val_clf,        rf_preds_val),
        "AUC-ROC":   roc_auc_score(y_val_clf,   rf_proba_val)
    },
    {
        "Model":     "Neural Network",
        "Accuracy":  nn_accuracy,
        "Precision": nn_precision,
        "Recall":    nn_recall,
        "F1-Score":  nn_f1,
        "AUC-ROC":   nn_auc
    }
]).round(4)

comparison_df

# Part 3: Model Evaluation & Interpretation (35 marks)

## Comprehensive Evaluation (15 marks)

In [ ]:
log_test_preds  = log_reg.predict(X_test_scaled)
log_test_proba  = log_reg.predict_proba(X_test_scaled)[:, 1]
rf_test_preds   = rf_clf.predict(X_test_scaled)
rf_test_proba   = rf_clf.predict_proba(X_test_scaled)[:, 1]

X_test_tensor   = torch.tensor(X_test_scaled.values, dtype=torch.float32)
model.eval()
with torch.no_grad():
    nn_test_proba = torch.sigmoid(model(X_test_tensor)).numpy()
    nn_test_preds = (nn_test_proba >= 0.5).astype(int)

clf_summary = pd.DataFrame([
    {
        "Model":     "Logistic Regression",
        "Accuracy":  accuracy_score(y_test_clf,  log_test_preds),
        "Precision": precision_score(y_test_clf, log_test_preds),
        "Recall":    recall_score(y_test_clf,    log_test_preds),
        "F1-Score":  f1_score(y_test_clf,        log_test_preds),
        "AUC-ROC":   roc_auc_score(y_test_clf,   log_test_proba)
    },
    {
        "Model":     "Random Forest Classifier",
        "Accuracy":  accuracy_score(y_test_clf,  rf_test_preds),
        "Precision": precision_score(y_test_clf, rf_test_preds),
        "Recall":    recall_score(y_test_clf,    rf_test_preds),
        "F1-Score":  f1_score(y_test_clf,        rf_test_preds),
        "AUC-ROC":   roc_auc_score(y_test_clf,   rf_test_proba)
    },
    {
        "Model":     "Neural Network",
        "Accuracy":  accuracy_score(y_test_clf,  nn_test_preds),
        "Precision": precision_score(y_test_clf, nn_test_preds),
        "Recall":    recall_score(y_test_clf,    nn_test_preds),
        "F1-Score":  f1_score(y_test_clf,        nn_test_preds),
        "AUC-ROC":   roc_auc_score(y_test_clf,   nn_test_proba)
    }
]).round(4)

lr_test_preds          = lr.predict(X_test_scaled)
rf_reg_test_preds      = rf_reg.predict(X_test_scaled)
best_rf_reg_test_preds = best_rf_reg.predict(X_test_scaled)

reg_summary = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "MAE":   mean_absolute_error(y_test_reg, lr_test_preds),
        "RMSE":  np.sqrt(mean_squared_error(y_test_reg, lr_test_preds)),
        "R²":    r2_score(y_test_reg, lr_test_preds)
    },
    {
        "Model": "Random Forest Regressor",
        "MAE":   mean_absolute_error(y_test_reg, rf_reg_test_preds),
        "RMSE":  np.sqrt(mean_squared_error(y_test_reg, rf_reg_test_preds)),
        "R²":    r2_score(y_test_reg, rf_reg_test_preds)
    },
    {
        "Model": "Tuned Random Forest Regressor",
        "MAE":   mean_absolute_error(y_test_reg, best_rf_reg_test_preds),
        "RMSE":  np.sqrt(mean_squared_error(y_test_reg, best_rf_reg_test_preds)),
        "R²":    r2_score(y_test_reg, best_rf_reg_test_preds)
    }
]).round(4)

print("CLASSIFICATION SUMMARY")
print("=" * 60)
display(clf_summary)

print("REGRESSION SUMMARY")
print("=" * 60)
display(reg_summary)

### b) For classification models: plot ROC curves for all models on the same figure, and plot a confusion matrix for the best model 

In [ ]:
from sklearn.metrics import roc_curve

fpr_log, tpr_log, _ = roc_curve(y_test_clf, log_test_proba)
fpr_rf,  tpr_rf,  _ = roc_curve(y_test_clf, rf_test_proba)
fpr_nn,  tpr_nn,  _ = roc_curve(y_test_clf, nn_test_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr_log, tpr_log, label="Logistic Regression")
plt.plot(fpr_rf,  tpr_rf,  label="Random Forest Classifier")
plt.plot(fpr_nn,  tpr_nn,  label="Neural Network")
plt.plot([0, 1],  [0, 1],  linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — Classification Models")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test_clf, nn_test_preds)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Low Tip", "High Tip"])
disp.plot()
plt.title("Confusion Matrix : Neural Network")
plt.tight_layout()
plt.show()

### c) For regression models: create a scatter plot of predicted vs actual tip amounts for the best model 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

rf_reg_test_preds = rf_reg.predict(X_test_scaled)

sample = min(5000, len(y_test_reg))
idx = np.random.choice(len(y_test_reg), sample, replace=False)

ax.scatter(
    y_test_reg.iloc[idx],
    rf_reg_test_preds[idx],
    alpha=0.3, s=10
)
max_val = max(y_test_reg.max(), rf_reg_test_preds.max())
ax.plot([0, max_val], [0, max_val], "r--", label="Perfect prediction")
ax.set_xlabel("Actual Tip Amount ($)")
ax.set_ylabel("Predicted Tip Amount ($)")
ax.set_title("Predicted vs Actual — Random Forest Regressor (Test Set)")
ax.legend()
plt.tight_layout()
plt.show()

### d) Perform a residual analysis for the best regression model (plot residual distribution and residuals vs predicted values) 

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

residuals = y_test_reg.values - rf_reg_test_preds

# Residual distribution
axes[0].hist(residuals, bins=100, color="steelblue", edgecolor="white")
axes[0].axvline(0, color="red", linestyle="--", label="Zero residual")
axes[0].set_xlabel("Residual ($)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Residual Distribution — RF Regressor")
axes[0].legend()

# Residuals vs predicted
axes[1].scatter(rf_reg_test_preds[idx], residuals[idx], alpha=0.3, s=10, color="steelblue")
axes[1].axhline(0, color="red", linestyle="--", label="Zero residual")
axes[1].set_xlabel("Predicted Tip Amount ($)")
axes[1].set_ylabel("Residual ($)")
axes[1].set_title("Residuals vs Predicted — RF Regressor")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nResidual Stats:")
print(f"  Mean:   {residuals.mean():.4f}  (should be ≈ 0)")
print(f"  Std:    {residuals.std():.4f}")
print(f"  Min:    {residuals.min():.4f}")
print(f"  Max:    {residuals.max():.4f}")

## Feature Importance (10 marks)

### a) Extract and plot feature importances from the Random Forest model (bar chart, sorted) 

In [ ]:
clf_imp = pd.Series(rf_clf.feature_importances_, index=X_train_scaled.columns) \
            .sort_values(ascending=False)

top_n = 15
fig, ax = plt.subplots(figsize=(10, 6))
clf_imp.head(top_n).sort_values(ascending=True).plot(
    kind="barh", ax=ax, color="steelblue", edgecolor="white"
)
ax.set_title(f"Top {top_n} Feature Importances — RF Classifier (target: high_tip)")
ax.set_xlabel("Importance Score")
ax.bar_label(ax.containers[0], fmt="%.4f", padding=3, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
reg_imp = pd.Series(rf_reg.feature_importances_, index=X_train_scaled.columns) \
            .sort_values(ascending=False)

top_n = 15
fig, ax = plt.subplots(figsize=(10, 6))
reg_imp.head(top_n).sort_values(ascending=True).plot(
    kind="barh", ax=ax, color="steelblue", edgecolor="white"
)
ax.set_title(f"Top {top_n} Feature Importances — RF Regressor (target: tip_amount)")
ax.set_xlabel("Importance Score")
ax.bar_label(ax.containers[0], fmt="%.4f", padding=3, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
clf_imp = pd.Series(rf_clf.feature_importances_, index=X_train_scaled.columns) \
            .sort_values(ascending=False)

reg_imp = pd.Series(rf_reg.feature_importances_, index=X_train_scaled.columns) \
            .sort_values(ascending=False)

summary = pd.DataFrame({
    "Classifier Importance": clf_imp,
    "Regressor Importance":  reg_imp,
}).sort_values("Classifier Importance", ascending=False)

print(f"\n{'Rank':<5} {'Feature':<35} {'Classifier':>12} {'Regressor':>12}")
print("─" * 68)
for rank, (feat, row) in enumerate(summary.head(10).iterrows(), 1):
    print(f"{rank:<5} {feat:<35} {row['Classifier Importance']:>12.4f} {row['Regressor Importance']:>12.4f}")

### b) Extract and interpret coefficients from the Linear/Logistic Regression model

In [ ]:
# Linear Regression coefficients
lr_coef = pd.DataFrame({
    "Feature":     X_train_scaled.columns,
    "Coefficient": lr.coef_
}).assign(Abs_Coefficient=lambda df: df["Coefficient"].abs()) \
  .sort_values("Abs_Coefficient", ascending=False)

top_n = 15
plot_data = lr_coef.head(top_n).sort_values("Coefficient")

colors = ["tomato" if c < 0 else "steelblue" for c in plot_data["Coefficient"]]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_data["Feature"], plot_data["Coefficient"], color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title(f"Top {top_n} Linear Regression Coefficients (target: tip_amount)")
ax.set_xlabel("Coefficient Value  (features are standardised)")
ax.bar_label(ax.containers[0], fmt="%.4f", padding=3, fontsize=8)
plt.tight_layout()
plt.show()

print(f"\n{'Rank':<5} {'Feature':<35} {'Coefficient':>12} {'Direction':>10}")
print("─" * 65)
for rank, (_, row) in enumerate(lr_coef.head(10).iterrows(), 1):
    direction = "positive ↑" if row["Coefficient"] > 0 else "negative ↓"
    print(f"{rank:<5} {row['Feature']:<35} {row['Coefficient']:>12.4f} {direction:>10}")

In [ ]:
# Logistic Regression coefficients
log_coef = pd.DataFrame({
    "Feature":     X_train_scaled.columns,
    "Coefficient": log_reg.coef_[0]
}).assign(Abs_Coefficient=lambda df: df["Coefficient"].abs()) \
  .sort_values("Abs_Coefficient", ascending=False)

# Plot top 15
top_n = 15
plot_data = log_coef.head(top_n).sort_values("Coefficient")

colors = ["tomato" if c < 0 else "steelblue" for c in plot_data["Coefficient"]]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_data["Feature"], plot_data["Coefficient"], color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title(f"Top {top_n} Logistic Regression Coefficients (target: high_tip)")
ax.set_xlabel("Coefficient Value  (features are standardised — larger |value| = stronger effect)")
ax.bar_label(ax.containers[0], fmt="%.4f", padding=3, fontsize=8)
plt.tight_layout()
plt.show()

print(f"\n{'Rank':<5} {'Feature':<35} {'Coefficient':>12} {'Odds Ratio':>12} {'Direction':>10}")
print("─" * 75)
for rank, (_, row) in enumerate(log_coef.head(10).iterrows(), 1):
    odds = f"{pd.np.exp(row['Coefficient']):.4f}" if hasattr(pd, 'np') else f"{__import__('numpy').exp(row['Coefficient']):.4f}"
    direction = "positive ↑" if row["Coefficient"] > 0 else "negative ↓"
    print(f"{rank:<5} {row['Feature']:<35} {row['Coefficient']:>12.4f} {odds:>12} {direction:>10}")

### c) Use SHAP values to explain individual predictions for 3 sample trips 

In [ ]:
sample = X_test_scaled.sample(3, random_state=42)
top_features = feature_importances["Feature"].head(8).tolist()
explainer = shap.TreeExplainer(rf_reg)
shap_values = explainer.shap_values(sample)
sample[top_features]

## Written Analysis (10 marks)

### a) Which model performed best for each task and why you think this is the case 

#### Regression Tasks
The tuned Random Forest performed the best with the scores of MAE: 1.1994 (lowest), RMSE:2.3510 (lowest), 1.1994, R²:	0.6303 (highest).
This out performed all the other models, because tip amount has non-linear relationships with features like trip distance and duration, the Random Forest exceles at this because of decision trees, whereas Linear Regression is constrained to a straight-line fit

#### Classification Tasks
The Random Forest edges out on AUC-ROC (0.6232), which is the most meaningful metric here as it measures discriminative ability independent of the classification threshold. However, Neural Network got better scores in Accuracy, Recall, and F1.

This means that within all the tasks the model which is the all around best is the random forest


### b) What features are most predictive of tip amount and whether this aligns with your intuition 

Fare amount is what would alighn to my intuition

Because the first thing that comes to mind when I tip is the bill itself so i could tip as percentage of said bill, so the higher the bill the more I would tip. My intuition aligns to the actual data because the random forest regressor assigns fare_amount an importance of 0.7453 which supasses all the other features


### c) Limitations of your models (e.g., data leakage concerns, feature limitations, dataset biases) 

The major problem with this is that it only uses payment type from cards because cash transcations don't record tips. This makes the data very bias and does not paint a good picture of the all the persons who use the nyc taxi service.

### d) Potential improvements you would make given more time or data 

A lot of improvements could be made like using the full data set for the turning instead of juse 200,000 and adding more features engineering on zone/ locaion data.

### e) A brief discussion comparing the neural network approach to the traditional ML models for this particular problem


| Model                   | Accuracy | Precision | Recall | F1-Score | AUC-ROC |
|-------------------------|----------|-----------|--------|----------|---------|
| Logistic Regression     | 0.7713   | 0.7694    | 0.9979 | 0.8689   | 0.6093  |
| Random Forest           | 0.7715   | 0.7697    | 0.9976 | 0.8690   | **0.6232**  |
| Neural Network          | **0.7716** | 0.7695  | **0.9983** | **0.8691** | 0.6200  |

This table shows that the performace was marginal at best, nn did'nt outperform any of the traditional models outright in this assignment. Furthermore Random Forest outperformed it in most tests, making that the best choice and theres also the fact that it is way less costly on you system than nn.

### AI discloser: used github copilot and chat gpt